## Deliverable 1: Task 1 Problem Specification

**1. Input, Output, and Examples:**
*   **Input Space ($X$):** Binary vectors of length 2, representing two sensors: $\{0,1\}^2$.
*   **Output Space ($Y$):** A single binary value $\{0, 1\}$ representing the disagreement warning.
*   **Labelled Examples:** $(0,0) \rightarrow 0$, $(0,1) \rightarrow 1$, $(1,0) \rightarrow 1$, $(1,1) \rightarrow 0$.

**2. Linear Separability Explanation:**
If these four points are plotted on a 2D plane, the positive class points $(0,1)$ and $(1,0)$ sit on opposite corners of a square from the negative class points $(0,0)$ and $(1,1)$. It is geometrically impossible to draw a single straight decision boundary that separates the 0s from the 1s. 

**3. Linear Model Prediction:**
If trained with only a single affine transformation followed by a sigmoid output, the model will fail to learn the XOR function. It will likely collapse to predicting a constant 0.5 probability for all inputs to minimize overall error, unable to capture the non-linear boundary.

## Deliverable 2: Model Design and Validation Criteria (Task 2)

**Baseline Design:** 2 inputs $\rightarrow$ 2 hidden units $\rightarrow$ 1 output.

**1. Why is the hidden nonlinearity scientifically necessary here?**
A stack of affine layers without nonlinear activations collapses mathematically into a single affine map. Nonlinearity is required to distort the feature space so the XOR problem becomes linearly separable in the hidden layer.

**2. Why is sigmoid plus binary cross-entropy a sensible engineering pairing for the output?**
The target is a single binary yes/no answer. The sigmoid function squashes the affine output into a valid probability distribution $[0, 1]$, and binary cross-entropy perfectly measures the divergence between this predicted probability and the true binary label.

**3. Validation Criteria:**
*   The final scalar loss approaches $0$.
*   All four thresholded predictions exactly match the target labels.
*   The gradients (`parameter.grad`) are non-zero during training, indicating an active learning signal.

## Deliverable 3: LLM Prompts and Corrections (Task 3)

**Prompt Used:**
> Generate minimal PyTorch code for a 2-2-1 neural network to solve the XOR problem. Use the four standard XOR examples. Use a ReLU hidden activation, a linear output layer paired with BCEWithLogitsLoss, and random weight initialization. Train using full-batch SGD for 2000 steps. After training, report the final loss, all four probabilities, thresholded labels, and the first-layer weight gradient tensor (`parameter.grad`) after a backward pass. Set a random seed for reproducibility.

**Corrections Made to Generated Code:**
The LLM successfully generated the boilerplate. However, for the symmetry experiment (Task 4C), the LLM naturally defaulted to random initialization. I had to manually override its initialization step with `nn.init.zeros_` to test the scientific hypothesis regarding identical weights.

In [2]:
# Deliverable 4 & 5: Binary XOR Experiment, Loss, Prediction, Gradient, Symmetry, and Activations
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Dataset
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

# 2. Model Definition
class XORModel(nn.Module):
    def __init__(self, hidden_activation=nn.Sigmoid()):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.activation = hidden_activation
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        h = self.activation(self.hidden(x))
        return self.output(h)

# --- PART A & B: Basic Learning & Backprop Check ---
print("--- Basic Learning and Gradient Check ---")
torch.manual_seed(42)
model = XORModel()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=1.0)

# Check Initial Gradient
initial_loss = criterion(model(X), y)
initial_loss.backward()
print(f"Initial Gradient of W(1):\n{model.hidden.weight.grad}\n")

# Train
for step in range(2000):
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_logits = model(X)
    final_loss = criterion(final_logits, y)
    probs = torch.sigmoid(final_logits)
    preds = (probs > 0.5).float()

print(f"Final Loss: {final_loss.item():.4f}")
print("Final Probabilities:\n", probs.squeeze().numpy())
print("Thresholded Predictions:\n", preds.squeeze().numpy())

# --- PART C: Symmetry Experiment ---
print("\n--- Symmetry Experiment ---")
sym_model = XORModel()
nn.init.zeros_(sym_model.hidden.weight)
nn.init.zeros_(sym_model.hidden.bias)
nn.init.zeros_(sym_model.output.weight)
nn.init.zeros_(sym_model.output.bias)

sym_optimizer = optim.SGD(sym_model.parameters(), lr=1.0)
for step in range(5):
    sym_optimizer.zero_grad()
    loss = criterion(sym_model(X), y)
    loss.backward()
    sym_optimizer.step()

print("Hidden Weights after 5 steps (Identical Initialization):\n", sym_model.hidden.weight.data)

# --- PART D: Activation Experiment ---
print("\n--- Activation Experiment ---")
activations = {'Sigmoid': nn.Sigmoid(), 'Tanh': nn.Tanh(), 'ReLU': nn.ReLU()}
for name, act in activations.items():
    torch.manual_seed(101) 
    act_model = XORModel(hidden_activation=act)
    act_optimizer = optim.SGD(act_model.parameters(), lr=0.5)
    
    loss = criterion(act_model(X), y)
    loss.backward()
    grad_norm = torch.norm(act_model.hidden.weight.grad).item()
    act_optimizer.step()
    
    for _ in range(2000):
        act_optimizer.zero_grad()
        loss = criterion(act_model(X), y)
        loss.backward()
        act_optimizer.step()
        
    final_loss = criterion(act_model(X), y).item()
    preds = (torch.sigmoid(act_model(X)) > 0.5).float()
    correct = (preds == y).all().item()
    print(f"{name:>7} | Loss: {final_loss:.4f} | 4/4 Correct? {correct} | Early Grad Norm: {grad_norm:.4f}")

--- Basic Learning and Gradient Check ---
Initial Gradient of W(1):
tensor([[-0.0040, -0.0044],
        [ 0.0064,  0.0081]])

Final Loss: 0.0153
Final Probabilities:
 [0.01805099 0.9862131  0.9862114  0.01491243]
Thresholded Predictions:
 [0. 1. 1. 0.]

--- Symmetry Experiment ---
Hidden Weights after 5 steps (Identical Initialization):
 tensor([[0., 0.],
        [0., 0.]])

--- Activation Experiment ---
Sigmoid | Loss: 0.0431 | 4/4 Correct? True | Early Grad Norm: 0.0023
   Tanh | Loss: 0.0036 | 4/4 Correct? True | Early Grad Norm: 0.0453
   ReLU | Loss: 0.6931 | 4/4 Correct? False | Early Grad Norm: 0.0000


In [3]:
# Deliverable 4 & 5: Three-Class Extension Code and Validation
print("--- Three-Class Extension ---")

# Labels: 0=(0,0), 1=(0,1) or (1,0), 2=(1,1)
y_multi = torch.tensor([0, 1, 1, 2], dtype=torch.long)

class MultiClassXOR(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4) 
        self.activation = nn.Tanh()
        self.output = nn.Linear(4, 3) 

    def forward(self, x):
        h = self.activation(self.hidden(x))
        return self.output(h)

torch.manual_seed(42)
multi_model = MultiClassXOR()
multi_criterion = nn.CrossEntropyLoss()
multi_optimizer = optim.SGD(multi_model.parameters(), lr=1.0)

for step in range(2000):
    multi_optimizer.zero_grad()
    logits = multi_model(X)
    loss = multi_criterion(logits, y_multi)
    loss.backward()
    multi_optimizer.step()

with torch.no_grad():
    final_logits = multi_model(X)
    probs = torch.softmax(final_logits, dim=1)
    preds = torch.argmax(probs, dim=1)

print("Final Probabilities:\n", probs.numpy())
print("Predicted Classes:", preds.numpy())

# Numerical stability check
example_logits = final_logits[0].clone()
prob_standard = torch.softmax(example_logits, dim=0)
prob_shifted = torch.softmax(example_logits + 100, dim=0)

print("\nStandard Softmax:  ", prob_standard.numpy())
print("Softmax (+100):    ", prob_shifted.numpy())
print("Sum of vector:     ", torch.sum(prob_standard).item())

--- Three-Class Extension ---
Final Probabilities:
 [[9.9954307e-01 4.5698977e-04 2.2732984e-09]
 [2.3502628e-04 9.9956506e-01 1.9995960e-04]
 [2.9354155e-04 9.9949491e-01 2.1150953e-04]
 [1.1050550e-09 5.7542749e-04 9.9942458e-01]]
Predicted Classes: [0 1 1 2]

Standard Softmax:   [9.9954307e-01 4.5698977e-04 2.2732984e-09]
Softmax (+100):     [9.995431e-01 4.569902e-04 2.273307e-09]
Sum of vector:      1.0000001192092896
